# 02. Multi-Model Benchmark, Deep Learning & Explainable AI (XAI)
**Project:** End-to-End Customer Churn Intelligence & Predictive Analytics Pipeline Using Python  
**Candidate:** Nithishkumar A S (Register Number: P25CA0044)  
**Degree:** Master of Computer Applications (MCA), BIHER

---
## Objective:
1. Feature Engineering & Stratified Train/Test Split
2. Multi-Model Benchmarking (Logistic Regression, Random Forest, XGBoost, Deep Neural Network)
3. Model Diagnostics (ROC-AUC, Confusion Matrices, Learning Curves)
4. Local Feature Attribution (XAI) and Counterfactual What-If Simulation


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

print("Inference libraries loaded.")


## 1. Load Pretrained Models & Feature Artifacts


In [ ]:
models_dir = os.path.join("..", "models")
scaler = joblib.load(os.path.join(models_dir, "scaler.joblib"))
features = joblib.load(os.path.join(models_dir, "features.joblib"))
summary = joblib.load(os.path.join(models_dir, "metrics_summary.joblib"))

results_df = pd.DataFrame([
    {
        "Model": v["name"],
        "Accuracy (%)": round(v["accuracy"] * 100, 2),
        "ROC-AUC": round(v["auc"], 4),
        "Recall (%)": round(v["recall"] * 100, 2),
        "Precision (%)": round(v["precision"] * 100, 2),
        "F1-Score (%)": round(v["f1"] * 100, 2),
        "Brier Score": round(v["brier_score"], 4)
    }
    for k, v in summary["results"].items()
])
results_df


## 2. Global Feature Importance Ranking (XGBoost Gini Importance)


In [ ]:
xgb_imp = summary["xgb_importance"]
sorted_imp = sorted(xgb_imp.items(), key=lambda x: x[1], reverse=True)[:10]

plt.figure(figsize=(9, 5))
sns.barplot(x=[s[1] for s in sorted_imp], y=[s[0].replace('_', ' ') for s in sorted_imp], palette="crest")
plt.title("Top 10 Churn Predictors (XGBoost Gini Impurity Gain)", fontsize=12, fontweight='bold')
plt.xlabel("Normalized Importance Score")
plt.ylabel("Engineered Feature")
plt.show()


## 3. Deep Neural Network (MLP) Training Loss Convergence Curve


In [ ]:
loss_data = joblib.load(os.path.join(models_dir, "mlp_loss_history.joblib"))
losses = loss_data["loss_curve"]

plt.figure(figsize=(8.5, 4))
plt.plot(range(1, len(losses) + 1), losses, color='#7C3AED', linewidth=2.0)
plt.title("Deep Neural Network (MLP: 64 -> 32) Training Loss vs. Epochs", fontsize=12, fontweight='bold')
plt.xlabel("Epochs")
plt.ylabel("Cross-Entropy Loss (Adam Optimizer)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


## 4. Local Explainability (XAI) & Counterfactual Simulation Demo


In [ ]:
from src.explainability import explain_prediction, simulate_what_if_counterfactual
xgb_model = joblib.load(os.path.join(models_dir, "xgboost_model.joblib"))

sample_cust = {
    'customerID': 'CUST-NOTEBOOK-01',
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 4,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 89.0,
    'TotalCharges': 356.0
}

# 1. Local Explainability
exp = explain_prediction(sample_cust, xgb_model, scaler, features)
print(f"Base Predicted Churn Probability: {exp['churn_probability'] * 100:.1f}% ({exp['risk_tier']})")
print("\nTop Churn Drivers:")
for d in exp['top_churn_drivers']:
    print(f"  [+] {d['feature']}: impact = +{d['impact']:.3f}")

# 2. Counterfactual Simulation
sim = simulate_what_if_counterfactual(
    sample_cust,
    {'Contract': 'Two year', 'TechSupport': 'Yes'},
    xgb_model,
    scaler
)
print(f"\nCounterfactual Intervention Applied: Two-Year Contract + Dedicated Tech Support")
print(f"Simulated New Churn Probability : {sim['simulated_churn_prob'] * 100:.1f}%")
print(f"Risk Reduction Delta (ΔP)       : -{sim['absolute_reduction'] * 100:.1f}% ({sim['percentage_risk_reduction']:.1f}% relative reduction)")
print(f"Annual Recurring Revenue Saved   : ${sim['annual_revenue_preserved']:,.2f}")
